# External Validation

Validates regime labels against variables **outside** the clustering feature set:
- NBER recession dates (official business-cycle peaks/troughs)
- VIX-threshold baseline (dumb rule: high VIX = stress)

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score, cohen_kappa_score

STRESS = "Stressful"

In [ ]:
raw = pd.read_csv("market_features_weekly.csv", parse_dates=["Date"], index_col="Date")
gmm = pd.read_csv("gmm_regimes.csv", index_col=0)
hmm = pd.read_csv("hmm_regimes.csv", index_col=0)

df = raw.copy()
df["Regime_GMM"] = gmm["Regime_label"].values
df["Regime_HMM"] = hmm["Regime_label"].values

df.head()

## 1. NBER recession overlap

Recession windows (NBER peak-to-trough) overlapping our sample: 2001, 2007–2009 GFC, 2020 COVID.

In [ ]:
NBER_RECESSIONS = [
    ("2001-03-01", "2001-11-30", "Dot-com"),
    ("2007-12-01", "2009-06-30", "GFC"),
    ("2020-02-01", "2020-04-30", "COVID"),
]

def in_recession(date):
    for start, end, _ in NBER_RECESSIONS:
        if pd.Timestamp(start) <= date <= pd.Timestamp(end):
            return True
    return False

df["NBER_recession"] = df.index.map(in_recession)

def recession_stress_rate(regime_col):
    rec = df[df["NBER_recession"]]
    non = df[~df["NBER_recession"]]
    return pd.Series({
        "Recession weeks (% Stressful)": (rec[regime_col] == STRESS).mean(),
        "Non-recession weeks (% Stressful)": (non[regime_col] == STRESS).mean(),
        "Recession weeks (n)": len(rec),
    })

nber_table = pd.DataFrame({
    "GMM": recession_stress_rate("Regime_GMM"),
    "HMM": recession_stress_rate("Regime_HMM"),
})
print(nber_table.round(3))

In [ ]:
for start, end, name in NBER_RECESSIONS:
    sub = df.loc[start:end]
    gmm_stress = (sub["Regime_GMM"] == STRESS).mean()
    hmm_stress = (sub["Regime_HMM"] == STRESS).mean()
    print(f"{name} ({start[:7]} to {end[:7]}): GMM {gmm_stress:.1%} Stressful, HMM {hmm_stress:.1%} Stressful")

## 2. VIX-threshold baseline

Simple rule: `VIX > 20` labels a week as baseline "Stress". Compare agreement with model stress labels.

In [ ]:
VIX_THRESHOLD = 20

df["VIX_baseline_stress"] = (df["VIX"] > VIX_THRESHOLD).map({True: STRESS, False: "Non-Stress"})

def baseline_agreement(model_col):
    model_stress = (df[model_col] == STRESS)
    base_stress = df["VIX"] > VIX_THRESHOLD
    return pd.Series({
        "Agreement with VIX>20 (%)": (model_stress == base_stress).mean(),
        "Cohen's kappa (Stress vs rest)": cohen_kappa_score(model_stress, base_stress),
        "Model Stress recall vs baseline": (model_stress & base_stress).sum() / base_stress.sum(),
        "Model Stress precision vs baseline": (model_stress & base_stress).sum() / model_stress.sum(),
    })

baseline_table = pd.DataFrame({
    "GMM": baseline_agreement("Regime_GMM"),
    "HMM": baseline_agreement("Regime_HMM"),
})
print(baseline_table.round(3))

## 3. Summary

If VIX>20 agreement is high (~80%+), the unsupervised machinery may add limited value beyond a one-line rule. Low agreement means the models capture structure beyond a simple vol threshold.